# Machine Learning for Time Series Data in Python [Regression]

Run all cells in order in a fresh Python 3.11 kernel. Complete original datasets are pinned and SHA-256 verified. All original gist examples are preserved verbatim in nonexecuting appendices. Code and matching explanations were corrected with user approval. Stock examples are historical educational exercises, not investment advice. Same-day cross-company regression is distinguished from the past-only lag forecast. No keys or GPU. All writes use an owned temporary workspace.

In [ ]:
import os,sys,subprocess,hashlib,urllib.request,urllib.parse,warnings,random
from pathlib import Path
from tempfile import TemporaryDirectory
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','pandas==3.1.0','numpy==2.3.5','scipy==1.16.3','matplotlib==3.10.8','statsmodels==0.14.6','pmdarima==2.1.1','scikit-learn==1.9.1','joblib==1.6.0','certifi==2026.10.23'],check=True)
import certifi
os.environ['SSL_CERT_FILE']=certifi.where()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
np.random.seed(42);random.seed(42)
pd.set_option('display.max_rows',12);pd.set_option('display.max_columns',12)
plt.rcParams.update({'figure.dpi':80,'figure.figsize':(10,5),'font.size':10})
workspace=TemporaryDirectory(prefix='tdb-time-series-11-');work=Path(workspace.name);previous_cwd=Path.cwd()
source_revision='65d4ae5d3436be3c0a921a3dcbda1b52e3ac8597'
source_files={'Arima Models in Python/candy_production.csv': '6b09ba45d482a23459499b98006d2e4911f31a06ded14ac2a1192dd5279c9056', 'Arima Models in Python/amazon_close.csv': 'f73498d0e1efa5cf9936bd4d2f810259c67ef39b3436f406b194de778fdf40ec', 'Arima Models in Python/co2.csv': 'a777e6a6000ff2dbf9bc12ff652a73a59b3d9c9e459f4330c3fd2574c6768b5c', 'Arima Models in Python/earthquakes.csv': '0d6e92b04f0337d13ea6055785032189763652ee4ec4bffbd80961964d6bbff9', 'Machine Learning For Time Series/prices.csv': 'd6fbdd4831ec8fd5884a690740d38bfc9dff44cfde0decc78ba122e70f58ebf6', 'Machine Learning For Time Series/preprocessed_prices.csv': '23e45fb2fbba7cc93ede87805546df86cba5531e82cc8e3cbdb4d63ddab3b176'}
for relative,digest in source_files.items():
    url='https://raw.githubusercontent.com/youssefHosni/Time-Series-Analysis/'+source_revision+'/'+urllib.parse.quote(relative)
    with urllib.request.urlopen(url,timeout=120)as response:payload=response.read()
    assert hashlib.sha256(payload).hexdigest()==digest,relative
    (work/Path(relative).name).write_bytes(payload)
os.chdir(work)
def checked_fit(model):
    result=model.fit(disp=False,maxiter=1000)
    if not result.mle_retvals.get('converged',False):
        result=model.fit(start_params=result.params,method='powell',disp=False,maxiter=2000)
    assert result.mle_retvals.get('converged',False),'Model did not converge'
    assert np.isfinite(result.params).all() and np.isfinite(result.aic)
    return result
print('Python:',sys.version.split()[0],'| pandas:',pd.__version__,'| NumPy:',np.__version__)
print('Six complete original datasets verified at pinned revision. No API keys or GPU.')

In [ ]:
# Load the New York stock exchange prices 
prices = pd.read_csv('prices.csv', 
            index_col='date',
            parse_dates=True).sort_index()

display(prices.head())
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 1 completed.')

In [ ]:
# change the plot style into fivethirtyeight 
plt.style.use('fivethirtyeight')

# Plot and show the time series on axis ax1
fig, ax1 = plt.subplots()
prices['close'].plot(ax=ax1, figsize=(12,10))
plt.title('New York stock prices change')
plt.xlabel('Date')
plt.ylabel('Stock prices')
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 2 completed.')

In [ ]:
print(prices.dtypes)
prices.index=pd.to_datetime(prices.index)
assert prices.index.is_monotonic_increasing
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 3 completed.')

In [ ]:
# Load the data
preprocessed_prices = pd.read_csv('preprocessed_prices.csv', parse_dates=True, index_col='date').sort_index()
preprocessed_prices
preprocessed_prices[['YHOO','EBAY']].plot(title='Yahoo and eBay closing prices');plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 4 completed.')

In [ ]:
# Scatterplot with one company per axis
preprocessed_prices.plot.scatter('EBAY', 'YHOO')
plt.title('Scatter plot of Yahoo and Ebay')
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 5 completed.')

In [ ]:
fig,ax=plt.subplots()
scatter=ax.scatter(preprocessed_prices['EBAY'],preprocessed_prices['YHOO'],c=np.arange(len(preprocessed_prices)),cmap='viridis')
fig.colorbar(scatter,ax=ax,label='Observation number (time order)')
ax.set(xlabel='EBAY',ylabel='YHOO',title='Time-colored closing-price association');plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 6 completed.')

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score,TimeSeriesSplit
from sklearn.metrics import r2_score
X_df=preprocessed_prices[['EBAY','NVDA','YHOO']].copy()
y_series=preprocessed_prices['AAPL'].copy()
X=X_df.copy();y=y_series.copy()
scores=cross_val_score(Ridge(),X,y,cv=TimeSeriesSplit(n_splits=3))
print('Forward-fold same-day association R²:',scores)
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 7 completed.')

In [ ]:
split=int(.8*len(X_df))
X_train,X_test=X_df.iloc[:split],X_df.iloc[split:]
y_train,y_test=y_series.iloc[:split],y_series.iloc[split:]
model=Ridge().fit(X_train,y_train)
predictions=model.predict(X_test)
holdout_r2=r2_score(y_test,predictions)
print('Chronological same-day association holdout R²:',holdout_r2)
comparison=pd.DataFrame({'True value':y_test,'Prediction':predictions},index=y_test.index)
comparison.plot(figsize=(12,5),title='Apple true and predicted closing price');plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 8 completed.')

In [ ]:
alphas=[.1,1e2,1e3]
fig,ax=plt.subplots(figsize=(12,5))
ax.plot(y_test.index,y_test,label='True values',color='k',alpha=.4)
for ii,alpha in enumerate(alphas):
    y_predicted=Ridge(alpha=alpha).fit(X_train,y_train).predict(X_test)
    ax.plot(y_test.index,y_predicted,color=plt.cm.viridis(ii/len(alphas)),label=f'alpha={alpha:g}')
ax.legend();ax.set(xlabel='Time',title='Ridge predictions across alpha values');plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 9 completed.')

In [ ]:
def remove_n_consecutive_rows(frame,n,percent):
    result=frame.copy()
    count=int(percent/100*len(frame)/n)
    chunks=[list(range(i,i+n+2))for i in range(len(frame)-n-1)]
    for _ in range(count):
        if not chunks:raise ValueError('Cannot place requested nonoverlapping blocks')
        indices=random.choice(chunks)
        result.iloc[indices[1:-1]]=np.nan
        selected=set(indices)
        chunks=[c for c in chunks if selected.isdisjoint(c)]
    return result
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 10 completed.')

In [ ]:
# The AIG data without missing values 
AIG = pd.DataFrame(preprocessed_prices['AIG'])
AIG.plot(figsize=(8,8))

plt.title('The stock prices of the AIG company')
plt.xlabel('Time (Years)')
plt.ylabel('Stock price')
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 11 completed.')

In [ ]:
# plotting the missing data
AIG_missing_data = remove_n_consecutive_rows(AIG, 100, 20)

AIG_missing_data.plot(figsize=(8,8))
plt.title('The stock prices of the AIG company with missing data')
plt.xlabel('Time (Years)')
plt.ylabel('Stock price')
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 12 completed.')

In [ ]:
# Interpolation in Pandas

# Return a boolean that notes where missing values are
missing_index = AIG_missing_data.isna()

# Interpolate linearly within missing windows
AIG_interp = AIG_missing_data.interpolate('linear')

# Plot the interpolated data in red and the data w/ missing values in black
ax = AIG_interp.plot(c='r')
AIG_missing_data.plot(c='k',ax=ax, lw=2)
ax.legend(['Interpolated','Observed with gaps'])
ax.set(xlabel='Time (Years)')
ax.set(ylabel='Stock value')
display(ax.set(title='AIG stock price value with missing and interpolated values'))
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 13 completed.')

In [ ]:
#Transforming to percent change with Pandas

def percent_change(values):
    """Calculates the % change between the last value
    and the mean of previous values"""
    # Separate the last value and all previous values into variables
    previous_values = values[:-1]
    last_value = values[-1]
    # Calculate the % difference between the last value
    # and the mean of earlier values
    percent_change = (last_value - np.mean(previous_values)) \
    / np.mean(previous_values)
    return percent_change
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 14 completed.')

In [ ]:
# Applying the transformation to our data

# Plot the raw data
fig, axs = plt.subplots(1, 2, figsize=(20, 10))

# plot the AIG with interplotation
axs[0].plot(AIG_interp, label='AIG')
axs[0].legend()

# Calculate % change and plot
AIG_perc_change = AIG_interp.rolling(window=20).apply(percent_change,raw=True)

# plot the trasnfoemd AIG
axs[1].plot(AIG_perc_change,label= 'AIG_transfomred')
axs[1].legend()

# set the title and x-axis and y-axis labels
axs[0].set(xlabel='Time (Years)')
axs[1].set(xlabel='Time (Years)')
axs[0].set(ylabel='Stock market value')
axs[1].set(ylabel='Percentage chnage in the stock market value')
plt.suptitle('AIG stock prices vs percentage change in stock prices')

for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 15 completed.')

In [ ]:
# Plotting a threshold on our data

fig, axs = plt.subplots(1, 2, figsize=(20, 10))
legends = ['AIG','AIG_transfromed']
for data, ax, l in zip([AIG_interp, AIG_perc_change], axs, legends):
    # Calculate the mean / standard deviation for the data
    data_mean = data.mean()
    data_std = data.std()
    # Plot the data, with a window that is 3 standard deviations
    # around the mean
    ax.plot(data, label=l)
    ax.legend()
    ax.axhline(data_mean.iloc[0] + data_std.iloc[0] * 3, ls='--', c='r')
    ax.axhline(data_mean.iloc[0] - data_std.iloc[0] * 3, ls='--', c='r')

# set the title and x-axis and y-axis labels
axs[0].set(xlabel='Time (Years)')
axs[1].set(xlabel='Time (Years)')
axs[0].set(ylabel='Stock market value')
axs[1].set(ylabel='Percentage chnage in the stock market value')
plt.suptitle('AIG stock prices vs percentage change in stock prices with applied threshold')
    
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 16 completed.')

In [ ]:
AIG_outlier_centered=AIG_perc_change-AIG_perc_change.mean()
std=AIG_perc_change.std()
outliers=np.abs(AIG_outlier_centered)>(std*3)
remaining=AIG_outlier_centered.mask(outliers)
replacement=float(remaining.median().iloc[0])
AIG_outlier_fixed=AIG_outlier_centered.mask(outliers,replacement)
print('Outliers replaced:',int(outliers.sum().sum()),'replacement:',replacement)
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 17 completed.')

In [ ]:
fig, axs = plt.subplots(1, 2,sharey=True,figsize=(20, 10))
axs[0].plot(AIG_outlier_centered, label='AIG with outliers')
axs[1].plot(AIG_outlier_fixed, label='AIG without outliers')

axs[0].legend()
axs[1].legend()

axs[0].set(xlabel='Time (Years)')
axs[1].set(xlabel='Time (Years)')
plt.suptitle('AIG stock prices with outliers Vs AIG stock prices without outliers ')
display(axs[0].set(ylabel='Percentage chnage in the stock market value'))
    
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 18 completed.')

In [ ]:
feats=preprocessed_prices.rolling(20).agg(['std','max']).dropna()
print(feats.head())
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 19 completed.')

In [ ]:
feats['AAPL'].plot(figsize=(10, 10))
plt.xlabel('Date [Years] ')
plt.ylabel('Stock price')
plt.title('The std Vs max of the stock price change of Apple')
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 20 completed.')

In [ ]:
# If we just take the mean, it returns a single value
a = np.array([[0, 1, 2], [0, 1, 2], [0, 1, 2]])
print(np.mean(a))
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 21 completed.')

In [ ]:
# We can use the partial function to initialize np.mean
# with an axis parameter
from functools import partial
mean_over_first_axis = partial(np.mean, axis=0)
print(mean_over_first_axis(a))
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 22 completed.')

In [ ]:
# Percentiles summarize your data
print(np.percentile(np.linspace(0, 200), q=20))
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 23 completed.')

In [ ]:
# Combining np.percentile() with partial functions to calculate a range of percentiles and apply it on toy data
data = np.linspace(0, 100)
# Create a list of functions using a list comprehension
percentile_funcs = [partial(np.percentile, q=ii) for ii in [20, 40, 60]]
# Calculate the output of each function in the same way
percentiles = [i_func(data) for i_func in percentile_funcs]
print(percentiles)
for func,q in zip(percentile_funcs,[20,40,60]):func.__name__=f'percentile_{q}'
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 24 completed.')

In [ ]:
# Calculate multiple percentiles of a rolling window on our prices time series
display(preprocessed_prices.rolling(20).aggregate(percentile_funcs).dropna())
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 25 completed.')

In [ ]:
# datetime features using Pandas
# Ensure our index is datetime
preprocessed_prices.index = pd.to_datetime(preprocessed_prices.index)

# Extract datetime features
day_of_week_num = preprocessed_prices.index.day_of_week
print('Days of the week in numbers:', day_of_week_num[:10])

day_of_week = preprocessed_prices.index.day_name()
print('Days of the week in names:', day_of_week[:10])
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 26 completed.')

In [ ]:
data=preprocessed_prices['AIG'].copy()
many_shifts=pd.DataFrame({f'lag_{lag}':data.shift(lag)for lag in range(1,8)})
lagged_frame=many_shifts.join(data.rename('target')).dropna()
many_shifts=lagged_frame.drop(columns='target')
lag_target=lagged_frame['target']
print(many_shifts.head(15))
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 27 completed.')

In [ ]:
lag_split=int(.8*len(many_shifts))
model=Ridge().fit(many_shifts.iloc[:lag_split],lag_target.iloc[:lag_split])
lag_predictions=model.predict(many_shifts.iloc[lag_split:])
lag_r2=r2_score(lag_target.iloc[lag_split:],lag_predictions)
print('Past-only AIG lag forecast holdout R²:',lag_r2)
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 28 completed.')

In [ ]:
# Visualize the fit model coefficients
fig, ax = plt.subplots(figsize=(10,10))
ax.bar(many_shifts.columns, model.coef_)
ax.set(xlabel='Coefficient name', ylabel='Coefficient value')
# Set formatting so it looks nice
plt.setp(ax.get_xticklabels(), rotation=45, horizontalalignment='right')
plt.title('Model coefficients for each lag')
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 29 completed.')

In [ ]:
from sklearn.model_selection import KFold
for tr,tt in KFold(n_splits=5).split(X_df):
    print('Last training index:',tr.max(),'first test index:',tt.min(),'contains future training:',bool(tr.max()>tt.min()))
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 30 completed.')

In [ ]:
from sklearn.model_selection import ShuffleSplit
def visualize_predictions(records,title):
    fig,axes=plt.subplots(2,1,figsize=(12,7))
    offset=0
    for i,(prediction,indices) in enumerate(records):
        axes[0].scatter(indices,prediction,s=7,label=f'Fold {i+1}')
        axes[1].scatter(np.arange(offset,offset+len(indices)),prediction,s=7)
        offset+=len(indices)
    axes[0].set(xlabel='Original time index',ylabel='Predicted AAPL')
    axes[1].set(xlabel='Prediction number',ylabel='Predicted AAPL')
    fig.suptitle(title);plt.tight_layout();plt.show()

X=X_df.to_numpy();y=y_series.to_numpy()
results=[]
for tr,tt in ShuffleSplit(n_splits=10,random_state=1).split(X,y):
    fitted=Ridge().fit(X[tr],y[tr])
    results.append((fitted.predict(X[tt]),tt))
display(visualize_predictions(results,'Shuffled counterexample — not forward forecasting'))
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 31 completed.')

In [ ]:
results=[]
for tr,tt in KFold(n_splits=10,shuffle=False).split(X,y):
    fitted=Ridge().fit(X[tr],y[tr])
    results.append((fitted.predict(X[tt]),tt))
display(visualize_predictions(results,'Unshuffled KFold — still trains on future blocks'))
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 32 completed.')

In [ ]:
# Import TimeSeriesSplit
from sklearn.model_selection import TimeSeriesSplit

# Create time-series cross-validation object
cv = TimeSeriesSplit(n_splits=10)

# Iterate through CV splits
fig, ax = plt.subplots()
for ii, (tr, tt) in enumerate(cv.split(X, y)):
    # Plot the training data on each iteration, to see the behavior of the CV
    ax.plot(tr, ii + y[tr]/1000)
    
ax.set(title='Training data on each CV iteration', ylabel='CV iteration')
ax.set(xlabel='time')
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 33 completed.')

In [ ]:
from sklearn.utils import resample
cv_coefficients=np.vstack([Ridge().fit(X[tr],y[tr]).coef_ for tr,tt in TimeSeriesSplit(n_splits=10).split(X,y)])
n_boots=100
bootstrap_means=np.vstack([resample(cv_coefficients,random_state=ii).mean(axis=0)for ii in range(n_boots)])
percentiles=np.percentile(bootstrap_means,[2.5,97.5],axis=0)
fig,ax=plt.subplots(figsize=(10,5))
ax.scatter(X_df.columns,percentiles[0],marker='_',s=200)
ax.scatter(X_df.columns,percentiles[1],marker='_',s=200)
ax.set(title='Descriptive resampling interval for mean fold coefficients');plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 34 completed.')

In [ ]:
# Model performance over time

# score function will be the correlation between the predicted and the true values
def my_corrcoef(est, X, y):
    """Return the correlation coefficient
    between model predictions and a validation set."""
    score = np.corrcoef(np.asarray(y).ravel(),np.asarray(est.predict(X)).ravel())[0,1]
    return score

# define the cv split and the regression model 
cv = TimeSeriesSplit(n_splits=100)
model = Ridge()
first_indices = []

# Grab the date of the first index of each validation set
for tr, tt in cv.split(X, y):
    # Fit the model on training data
    first_indices.append(X_df.index[tt[0]])
    
# Calculate the CV scores and convert to a Pandas Series
cv_scores = cross_val_score(model, X, y, cv=cv, scoring = my_corrcoef)
cv_scores = pd.DataFrame(cv_scores, index=first_indices)

# Visualizing model scores as a timeseries
fig, axs = plt.subplots(2, 1, figsize=(20, 20), sharex=False)

# Calculate a rolling mean of scores over time
cv_scores_mean = cv_scores.rolling(10, min_periods=1).mean()
cv_scores_mean.plot(ax=axs[0])
axs[0].set(title='Validation scores (correlation)', ylim=[-1, 1])

# Plot the raw data
X_df.plot(ax=axs[1])
axs[1].set(title='Validation data')
expanding_scores=cv_scores.copy()
print('Expanding-window score range:',float(cv_scores.min().iloc[0]),float(cv_scores.max().iloc[0]))
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 35 completed.')

In [ ]:
# Only keep the last 100 datapoints in the training data
window = 100

# Initialize the CV with this window size
cv = TimeSeriesSplit(n_splits=10, max_train_size=window)

model = Ridge()
first_indices = []

for tr, tt in cv.split(X, y):
    # Fit the model on training data
    first_indices.append(X_df.index[tt[0]])
    

model = Ridge()   
cv_scores = cross_val_score(model, X, y, cv=cv, scoring = my_corrcoef)

# Calculate the CV scores and convert to a Pandas Series
cv_scores = pd.DataFrame(cv_scores, index=first_indices)

#Visualizing model scores as a timeseries
fig, axs = plt.subplots(2, 1, figsize=(20, 20), sharex=False)

# Calculate a rolling mean of scores over time
cv_scores_mean = cv_scores.rolling(10, min_periods=1).mean()
cv_scores_mean.plot(ax=axs[0])
axs[0].set(title='Validation scores (correlation)', ylim=[-1, 1])

# Plot the raw data
X_df.plot(ax=axs[1])
axs[1].set(title='Validation data')
window_scores=cv_scores.copy()
print('One-hundred-row window score range:',float(cv_scores.min().iloc[0]),float(cv_scores.max().iloc[0]))
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 36 completed.')

In [ ]:
from numpy.testing import assert_allclose
def independent_ridge(train_x,train_y,test_x,alpha=1.):
    train_x=np.asarray(train_x);train_y=np.asarray(train_y).ravel();test_x=np.asarray(test_x)
    mean_x=train_x.mean(axis=0);mean_y=train_y.mean();xc=train_x-mean_x;yc=train_y-mean_y
    coefficients=np.linalg.solve(xc.T@xc+alpha*np.eye(xc.shape[1]),xc.T@yc)
    return (test_x-mean_x)@coefficients+mean_y,coefficients
assert len(prices)==84912 and preprocessed_prices.shape==(1278,46)
expected,coefficients=independent_ridge(X_train,y_train,X_test)
assert_allclose(predictions,expected,rtol=1e-10,atol=1e-8)
expected_r2=1-np.sum((y_test.to_numpy()-expected)**2)/np.sum((y_test.to_numpy()-y_test.mean())**2)
assert_allclose(holdout_r2,expected_r2,atol=1e-10)
assert np.array_equal(AIG['AIG'].to_numpy(),preprocessed_prices['AIG'].to_numpy())
assert int(AIG_missing_data.isna().sum().iloc[0])==200
observed=np.flatnonzero(AIG_missing_data.iloc[:,0].notna())
interpolated=np.interp(np.arange(len(AIG)),observed,AIG_missing_data.iloc[observed,0].to_numpy())
assert_allclose(AIG_interp.iloc[:,0],interpolated,atol=1e-12)
windows=np.lib.stride_tricks.sliding_window_view(interpolated,20)
expected_change=(windows[:,-1]-windows[:,:-1].mean(axis=1))/windows[:,:-1].mean(axis=1)
assert_allclose(AIG_perc_change.iloc[19:,0],expected_change,atol=1e-12)
assert_allclose(replacement,np.nanmedian(AIG_outlier_centered.to_numpy()[~outliers.to_numpy()]),atol=1e-12)
assert_allclose(AIG_outlier_fixed.to_numpy()[outliers.to_numpy()],replacement)
matrix=preprocessed_prices.to_numpy();rolling_windows=np.lib.stride_tricks.sliding_window_view(matrix,20,axis=0)
for name,fn in [('std',lambda a:np.std(a,axis=-1,ddof=1)),('max',lambda a:np.max(a,axis=-1))]:
    # pandas' incremental rolling variance and fresh NumPy windows differ at floating-point roundoff scale.
    assert_allclose(feats.xs(name,axis=1,level=1),fn(rolling_windows),rtol=1e-10,atol=1e-8)
for lag in range(1,8):assert_allclose(many_shifts[f'lag_{lag}'],data.to_numpy()[7-lag:len(data)-lag],atol=1e-12)
assert many_shifts.index[0]==data.index[7]
expected_lag,_=independent_ridge(many_shifts.iloc[:lag_split],lag_target.iloc[:lag_split],many_shifts.iloc[lag_split:])
assert_allclose(lag_predictions,expected_lag,atol=1e-10)
for splits,actual in [(TimeSeriesSplit(n_splits=100),expanding_scores),(TimeSeriesSplit(n_splits=10,max_train_size=100),window_scores)]:
    independent_scores=[]
    for tr,tt in splits.split(X,y):
        assert tr.max()<tt.min()
        if splits.max_train_size is not None:assert len(tr)<=100
        pred,_=independent_ridge(X[tr],y[tr],X[tt])
        true=y[tt]-y[tt].mean();centered=pred-pred.mean()
        independent_scores.append(np.dot(true,centered)/np.sqrt(np.dot(true,true)*np.dot(centered,centered)))
    assert_allclose(actual.iloc[:,0],independent_scores,atol=1e-8)
assert any(tr.max()>tt.min()for tr,tt in KFold(10,shuffle=False).split(X))
independent_boots=np.vstack([cv_coefficients[np.random.RandomState(i).randint(0,len(cv_coefficients),len(cv_coefficients))].mean(axis=0)for i in range(100)])
assert_allclose(bootstrap_means,independent_boots,atol=1e-12)
assert_allclose(percentiles,np.percentile(independent_boots,[2.5,97.5],axis=0),atol=1e-12)
metrics={'long_price_rows':len(prices),'matrix_shape':list(preprocessed_prices.shape),'same_day_holdout_r2':float(holdout_r2),'past_only_lag_holdout_r2':float(lag_r2),'removed_observations':200,'forward_cv_folds':100,'fixed_window_cv_folds':10,'negative_expanding_scores':int((expanding_scores.iloc[:,0]<0).sum()),'bootstrap_resamples':100}
print('Independent Ridge/R², all interpolation/window/lag features, 110 forward-fold correlations and deterministic resampling checks passed.')
print('METRICS:',metrics)

os.chdir(previous_cwd)
workspace.cleanup()
print('Owned temporary workspace cleaned up.')

## Original source code

### Original block 1

```text
# Load the New York stock exchange prices 
prices = pd.read_csv('prices.csv', 
            index_col='date',
            parse_dates=True)

prices.head()

```

### Original block 2

```text
# change the plot style into fivethirtyeight 
plt.style.use('fivethirtyeight')

# Plot and show the time series on axis ax1
fig, ax1 = plt.subplots()
prices['close'].plot(ax=ax1, figsize=(12,10))
plt.title('New York stock prices change')
plt.xlabel('Date')
plt.ylabel('Stock prices')
plt.show()
```

### Original block 3

```text
# print the type of the data
prices.dtypes
# convert the type of the index from object to time
prices.index = pd.to_datetime(prices.index)

```

### Original block 4

```text
# Load the data
preprocessed_prices = pd.read_csv('preprocessed_prices.csv', parse_dates=True, index_col='date')
preprocessed_prices
```

### Original block 5

```text
# Scatterplot with one company per axis
preprocessed_prices.plot.scatter('EBAY', 'YHOO')
plt.title('Scatter plot of Yahoo and Ebay')
plt.show()
```

### Original block 6

```text
# Scatterplot with color relating to time
preprocessed_prices.plot.scatter('EBAY', 'YHOO', c=preprocessed_prices.index, 
                    cmap=plt.cm.viridis, colorbar=True)
plt.show()
```

### Original block 7

```text
#Fitting a simple regression model

from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# Use stock symbols to extract training data
X = preprocessed_prices[['EBAY', 'NVDA', 'YHOO']]
y = preprocessed_prices[['AAPL']]
# Fit and score the model with cross-validation
scores = cross_val_score(Ridge(), X, y, cv=3)
print(scores)
```

### Original block 8

```text
# Visualizing predicted values

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Split our data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=.8, shuffle=False, random_state=1)

# Fit our model and generate predictions
model = Ridge()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
score = r2_score(y_test, predictions)
print(score)

y_test['predictions'] = predictions
y_test.rename(columns= {'AAPL':'True_vlaue'}, inplace=True)

# Visualize our predictions along with the "true" values, and print the score
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(y_test['True_vlaue'], color='k', lw=3)
ax.plot(y_test['predictions'], color='r', lw=2)
ax.legend(['True values', 'Predicitions'])
plt.title('Apple stock price true value and predicted price')
plt.show()
```

### Original block 9

```text
alphas = [.1, 1e2, 1e3]
ax.plot(y_test, color='k', alpha=.3, lw=3)
for ii, alpha in enumerate(alphas):
y_predicted = Ridge(alpha=alpha).fit(X_train, y_train).predict(X_test)
ax.plot(y_predicted, c=cmap(ii / len(alphas)))
ax.legend(['True values', 'Model 1', 'Model 2', 'Model 3'])
ax.set(xlabel="Time")
```

### Original block 10

```text
# Create missing rows at random 
def remove_n_consecutive_rows(frame, n, percent):
    chunks_to_remove = int(percent/100*frame.shape[0]/n)
    #split the indices into chunks of length n+2
    chunks = [list(range(i,i+n+2)) for i in range(0, frame.shape[0]-n)]
    drop_indices = list()
    for i in range(chunks_to_remove):
        indices = random.choice(chunks)
        drop_indices+=indices[1:-1]
        #remove all chunks which contain overlapping values with indices
        chunks = [c for c in chunks if not any(n in indices for n in c)]
    #drop_indices = frame.index[drop_indices]    
    frame.iloc[drop_indices,] = np.nan
    return frame
```

### Original block 11

```text
# The AIG data without missing values 
AIG = pd.DataFrame(preprocessed_prices['AIG'])
AIG.plot(figsize=(8,8))

plt.title('The stock prices of the AIG company')
plt.xlabel('Time (Years)')
plt.ylabel('Stock price')
```

### Original block 12

```text
# plotting the missing data
AIG_missing_data = remove_n_consecutive_rows(AIG, 100, 20)

AIG_missing_data.plot(figsize=(8,8))
plt.title('The stock prices of the AIG company with missing data')
plt.xlabel('Time (Years)')
plt.ylabel('Stock price')
```

### Original block 13

```text
# Interpolation in Pandas

# Return a boolean that notes where missing values are
missing_index = AIG_missing_data.isna()

# Interpolate linearly within missing windows
AIG_interp = AIG_missing_data.interpolate('linear')

# Plot the interpolated data in red and the data w/ missing values in black
ax = AIG_interp.plot(c='r')
AIG_missing_data.plot(c='k',ax=ax, lw=2)
ax.legend(['AIG_missing','AIG_interpolated'])
ax.set(xlabel='Time (Years)')
ax.set(ylabel='Stock value')
ax.set(title='AIG stock price value with missing and interpolated values')
```

### Original block 14

```text
#Transforming to percent change with Pandas

def percent_change(values):
    """Calculates the % change between the last value
    and the mean of previous values"""
    # Separate the last value and all previous values into variables
    previous_values = values[:-1]
    last_value = values[-1]
    # Calculate the % difference between the last value
    # and the mean of earlier values
    percent_change = (last_value - np.mean(previous_values)) \
    / np.mean(previous_values)
    return percent_change
```

### Original block 15

```text
# Applying the transformation to our data

# Plot the raw data
fig, axs = plt.subplots(1, 2, figsize=(20, 10))

# plot the AIG with interplotation
axs[0].plot(AIG_interp, label='AIG')
axs[0].legend()

# Calculate % change and plot
AIG_perc_change = AIG_interp.rolling(window=20).aggregate(percent_change)

# plot the trasnfoemd AIG
axs[1].plot(AIG_perc_change,label= 'AIG_transfomred')
axs[1].legend()

# set the title and x-axis and y-axis labels
axs[0].set(xlabel='Time (Years)')
axs[1].set(xlabel='Time (Years)')
axs[0].set(ylabel='Stock market value')
axs[1].set(ylabel='Percentage chnage in the stock market value')
plt.suptitle('AIG stock prices vs percentage change in stock prices')

```

### Original block 16

```text
# Plotting a threshold on our data

fig, axs = plt.subplots(1, 2, figsize=(20, 10))
legends = ['AIG','AIG_transfromed']
for data, ax, l in zip([AIG_interp, AIG_perc_change], axs, legends):
    # Calculate the mean / standard deviation for the data
    data_mean = data.mean()
    data_std = data.std()
    # Plot the data, with a window that is 3 standard deviations
    # around the mean
    ax.plot(data, label=l)
    ax.legend()
    ax.axhline(data_mean[0] + data_std[0] * 3, ls='--', c='r')
    ax.axhline(data_mean[0] - data_std[0] * 3, ls='--', c='r')

# set the title and x-axis and y-axis labels
axs[0].set(xlabel='Time (Years)')
axs[1].set(xlabel='Time (Years)')
axs[0].set(ylabel='Stock market value')
axs[1].set(ylabel='Percentage chnage in the stock market value')
plt.suptitle('AIG stock prices vs percentage change in stock prices with applied threshold')
    
```

### Original block 17

```text
# Center the data so the mean is 0
AIGِ_outlier_centered = AIG_perc_change - AIG_perc_change.mean()

# Calculate standard deviation
std = AIG_perc_change.std()

# Use the absolute value of each datapoint
# to make it easier to find outliers
outliers = np.abs(AIGِ_outlier_centered) > (std * 3)

# Replace outliers with the median value
# We'll use np.nanmean since there may be nans around the outliers
AIG_outlier_fixed = AIGِ_outlier_centered.copy()
AIG_outlier_fixed[outliers] = np.nanmedian(AIG_outlier_fixed)
```

### Original block 18

```text
fig, axs = plt.subplots(1, 2,sharey=True,figsize=(20, 10))
axs[0].plot(AIGِ_outlier_centered, label='AIG with outliers')
axs[1].plot(AIG_outlier_fixed, label='AIG without outliers')

axs[0].legend()
axs[1].legend()

axs[0].set(xlabel='Time (Years)')
axs[1].set(xlabel='Time (Years)')
plt.suptitle('AIG stock prices with outliers Vs AIG stock prices without outliers ')
axs[0].set(ylabel='Percentage chnage in the stock market value')
    
```

### Original block 19

```text
# Calculate a rolling window, then extract two features
feats = preprocessed_prices.rolling(20).aggregate([np.std, np.max]).dropna()
feats.head(5)
```

### Original block 20

```text
feats['AAPL'].plot(figsize=(10, 10))
plt.xlabel('Date [Years] ')
plt.ylabel('Stock price')
plt.title('The std Vs max of the stock price change of Apple')
```

### Original block 21

```text
# If we just take the mean, it returns a single value
a = np.array([[0, 1, 2], [0, 1, 2], [0, 1, 2]])
print(np.mean(a))
```

### Original block 22

```text
# We can use the partial function to initialize np.mean
# with an axis parameter
from functools import partial
mean_over_first_axis = partial(np.mean, axis=0)
print(mean_over_first_axis(a))
```

### Original block 23

```text
# Percentiles summarize your data
print(np.percentile(np.linspace(0, 200), q=20))
```

### Original block 24

```text
# Combining np.percentile() with partial functions to calculate a range of percentiles and apply it on toy data
data = np.linspace(0, 100)
# Create a list of functions using a list comprehension
percentile_funcs = [partial(np.percentile, q=ii) for ii in [20, 40, 60]]
# Calculate the output of each function in the same way
percentiles = [i_func(data) for i_func in percentile_funcs]
print(percentiles)
```

### Original block 25

```text
# Calculate multiple percentiles of a rolling window on our prices time series
preprocessed_prices.rolling(20).aggregate(percentile_funcs).dropna()
```

### Original block 26

```text
# datetime features using Pandas
# Ensure our index is datetime
preprocessed_prices.index = pd.to_datetime(preprocessed_prices.index)

# Extract datetime features
day_of_week_num = preprocessed_prices.index.weekday
print('Days of the week in numbers:', day_of_week_num[:10])

day_of_week = preprocessed_prices.index.day_name()
print('Days of the week in names:', day_of_week[:10])
```

### Original block 27

```text
# create shifts in the data

# slice the AIG company data
data = preprocessed_prices['AIG']

# Shifts 
shifts = [1, 2, 3, 4, 5, 6, 7]

# Create a dictionary of time-shifted data
many_shifts = {'lag_{}'.format(ii): data.shift(ii) for ii in shifts}

# Convert the shifts into a dataframe
many_shifts = pd.DataFrame(many_shifts)
many_shifts.fillna(0, inplace=True)
many_shifts.head(15)
```

### Original block 28

```text
# Fit the model using these input features
model = Ridge()
model.fit(many_shifts, data)
```

### Original block 29

```text
# Visualize the fit model coefficients
fig, ax = plt.subplots(figsize=(10,10))
ax.bar(many_shifts.columns, model.coef_)
ax.set(xlabel='Coefficient name', ylabel='Coefficient value')
# Set formatting so it looks nice
plt.setp(ax.get_xticklabels(), rotation=45, horizontalalignment='right')
plt.title('Model coefficients for each lag')
```

### Original block 30

```text
from sklearn.model_selection import KFold
cv = KFold(n_splits=5)
for tr, tt in cv.split(X, y):
  ...
```

### Original block 31

```text
# Import ShuffleSplit and create the cross-validation object
from sklearn.model_selection import ShuffleSplit
cv = ShuffleSplit(n_splits=10, random_state=1)

# convert the dataframe into numpy array
X = X.to_numpy()
y = y.to_numpy()


# Iterate through CV splits
results = []
for tr, tt in cv.split(X, y):
    # Fit the model on training data
    model.fit(X[tr], y[tr])

    # Generate predictions on the test data, score the predictions, and collect
    prediction = model.predict(X[tt])
    score = r2_score(y[tt], prediction)
    results.append((prediction, score, tt))
    
# Custom function to quickly visualize predictions
visualize_predictions(results)
```

### Original block 32

```text
# Create KFold cross-validation object
from sklearn.model_selection import KFold
cv = KFold(n_splits=10, shuffle=False)

# Iterate through CV splits
results = []
for tr, tt in cv.split(X, y):
    # Fit the model on training data
    model.fit(X[tr],y[tr])
    
    # Generate predictions on the test data and collect
    prediction = model.predict(X[tt])
    results.append((prediction, tt))
    
# Custom function to quickly visualize predictions
visualize_predictions(results)
```

### Original block 33

```text
# Import TimeSeriesSplit
from sklearn.model_selection import TimeSeriesSplit

# Create time-series cross-validation object
cv = TimeSeriesSplit(n_splits=10)

# Iterate through CV splits
fig, ax = plt.subplots()
for ii, (tr, tt) in enumerate(cv.split(X, y)):
    # Plot the training data on each iteration, to see the behavior of the CV
    ax.plot(tr, ii + y[tr]/1000)
    
ax.set(title='Training data on each CV iteration', ylabel='CV iteration')
ax.set(xlabel='time')
plt.show()
```

### Original block 34

```text
# Bootstrapping the mean
from sklearn.utils import resample

# cv_coefficients has shape (n_cv_folds, n_coefficients)
n_coefficients = cv_coefficeints.shape[-1]
n_boots = 100
bootstrap_means = np.zeros((n_boots, n_coefficients))
                           
for ii in range(n_boots):
    # Generate random indices for our data with replacement,
    # then take the sample mean
    random_sample = resample(cv_coefficeints)
    bootstrap_means[ii] = random_sample.mean(axis=0)
    
# Compute the percentiles of choice for the bootstrapped means
percentiles = np.percentile(bootstrap_means, (2.5, 97.5), axis=0)

# Plotting the bootstrapped coefficients
fig, ax = plt.subplots(figsize=(10,5))
ax.scatter(X_df.columns, percentiles[0], marker='_', s=200)
ax.scatter(X_df.columns, percentiles[1], marker='_', s=200)
ax.set(title='95% confidence intervals for model coefficients')
```

### Original block 35

```text
# Model performance over time

# score function will be the correlation between the predicted and the true values
def my_corrcoef(est, X, y):
    """Return the correlation coefficient
    between model predictions and a validation set."""
    score = np.corrcoef(np.hstack((y, est.predict(X))))[1, 0]
    return score

# define the cv split and the regression model 
cv = TimeSeriesSplit(n_splits=100)
model = Ridge()
first_indices = []

# Grab the date of the first index of each validation set
for tr, tt in cv.split(X, y):
    # Fit the model on training data
    first_indices.append(X_df.index[tt[0]])
    
# Calculate the CV scores and convert to a Pandas Series
cv_scores = cross_val_score(model, X, y, cv=cv, scoring = my_corrcoef)
cv_scores = pd.DataFrame(cv_scores, index=first_indices)

# Visualizing model scores as a timeseries
fig, axs = plt.subplots(2, 1, figsize=(20, 20), sharex=False)

# Calculate a rolling mean of scores over time
cv_scores_mean = cv_scores.rolling(10, min_periods=1).mean()
cv_scores_mean.plot(ax=axs[0])
axs[0].set(title='Validation scores (correlation)', ylim=[0, 1])

# Plot the raw data
X_df.plot(ax=axs[1])
axs[1].set(title='Validation data')
```

### Original block 36

```text
# Only keep the last 100 datapoints in the training data
window = 100

# Initialize the CV with this window size
cv = TimeSeriesSplit(n_splits=10, max_train_size=window)

model = Ridge()
first_indices = []

for tr, tt in cv.split(X, y):
    # Fit the model on training data
    first_indices.append(X_df.index[tt[0]])
    

model = Ridge()   
cv_scores = cross_val_score(model, X, y, cv=cv, scoring = my_corrcoef)

# Calculate the CV scores and convert to a Pandas Series
cv_scores = pd.DataFrame(cv_scores, index=first_indices)

#Visualizing model scores as a timeseries
fig, axs = plt.subplots(2, 1, figsize=(20, 20), sharex=False)

# Calculate a rolling mean of scores over time
cv_scores_mean = cv_scores.rolling(10, min_periods=1).mean()
cv_scores_mean.plot(ax=axs[0])
axs[0].set(title='Validation scores (correlation)', ylim=[0, 1])

# Plot the raw data
X_df.plot(ax=axs[1])
axs[1].set(title='Validation data')
```